# 04 private test · bowen + 01c ASR -> `submission.jsonl`

`boweneval (3).ipynb` (03c repair) on `private_test`, with one added step: private has no transcripts, so the 01c model in `asr_model/` transcribes every clip first (sherpa-onnx greedy, 4 threads, `norm()` + `vi_itn.py`, the same code as the Docker's `predict.py`). `infer.py` is byte-identical to boweneval's. No held-out eval.

**Setup**
1. Settings: Accelerator **GPU T4 x2**, internet on.
2. Add Input: the private test dataset (the folder holding `private_test/`), `bowen` (02 output with `llm_model/` and `lifemate.py`), `thbased01` (01c output with `asr_model/` and `vi_itn.py`).

**Run**: `SMOKE = True` first (16 samples), then `SMOKE = False`, Save & Run All. About 3,650 clips of ASR, then about 2 h of LLM.

`TIME_BUDGET = 120` as in boweneval. The Docker stops at 20 s, so samples over 30 s here may differ from what the Docker produces.


In [ ]:
import glob
import json

import torch
assert torch.cuda.device_count() == 2, "Set Settings > Accelerator to GPU T4 x2"
print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(2)])
_config = json.load(open(glob.glob("/kaggle/input/**/llm_model/config.json", recursive=True)[0], encoding="utf-8"))
TRANSFORMERS = {"spark2_5": "4.57.1", "lfm3": "5.20.2"}.get(_config["model_type"], "5.22.1")  # the version each model was trained on
print(_config["model_type"], "transformers", TRANSFORMERS)
!pip install -q "transformers=={TRANSFORMERS}" "accelerate==1.15.0" "sherpa-onnx==2.0.1"


In [ ]:
%%writefile settings.py

import glob
from pathlib import Path

SMOKE = True  # True: 8 samples per GPU
SPLIT = "private_test"
DTYPE = "float16"
SHARDS = 2  # one process per T4
MAX_STEPS = 8  # gold never uses more than 6 calls
MAX_NEW_TOKENS = 320  # the longest gold call is 304 characters, plus a written reason when the model was trained with WHY
TIME_BUDGET = 120.0  # a file upload is not timed. The Docker must stay under the real limit of 30 s
STRIP_MARKS = True  # remove the tagger's [ ] from ASR text. False: the old behaviour
SEED = 42

ROOT = Path([p for p in glob.glob(f"/kaggle/input/**/{SPLIT}/{SPLIT}.jsonl", recursive=True) if "__MACOSX" not in p][0]).parents[1]
LLM = str(Path(glob.glob("/kaggle/input/**/llm_model/config.json", recursive=True)[0]).parent)
LIFEMATE_DIR = str(Path(LLM).parent)  # lifemate.py as saved with these weights
ASR_DIR = str(Path(glob.glob("/kaggle/input/**/asr_model/encoder.int8.onnx", recursive=True)[0]).parents[1])  # 01c output: asr_model/ and vi_itn.py
ASR_JSON = "/kaggle/working/asr_transcripts.json"  # written by the ASR cell below

In [ ]:
%%writefile infer.py
"""One shard of the agent: python infer.py <split> <gold|asr> <shard>. One process per split, each split has its own zaic_tools."""
import json
import re
import sys
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from transformers.integrations import sdpa_attention

from settings import *

sdpa_attention.use_gqa_in_sdpa = lambda *args, **kwargs: False  # a T4 has no grouped-key kernel: Qwen3.5 fell back to float32 math and ran out of memory on long prompts (public_0110, public_0157)

sys.path[:0] = [LIFEMATE_DIR, str(ROOT / sys.argv[1])]
import lifemate as lm
from zaic_tools import ToolEnv

split, mode, shard = sys.argv[1], sys.argv[2], int(sys.argv[3])
set_seed(SEED)
tok = AutoTokenizer.from_pretrained(LLM)
model = AutoModelForCausalLM.from_pretrained(LLM, dtype=DTYPE, device_map="auto" if torch.cuda.is_available() else None, trust_remote_code=True).eval()
stop_id = tok.convert_tokens_to_ids(lm.STOP)  # not every repo has a generation_config, so the default eos may never fire
catalog = lm.load_catalog(ROOT, split)
rows = lm.load_split(ROOT, split)
asr = json.load(open(ASR_JSON, encoding="utf-8")) if mode == "asr" else {}
if split == "train":
    rows = sum(lm.split_rows(rows)[1:], [])  # only held-out samples say anything
rows = rows[shard::SHARDS][: 8 if SMOKE else None]


@torch.inference_mode()
def generate(msgs):
    ids = tok(lm.prompt_text(msgs), return_tensors="pt", add_special_tokens=False).to(model.device)
    assert ids.input_ids.shape[1] + MAX_NEW_TOKENS <= 16384, "over the competition context limit"
    out = model.generate(**ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, eos_token_id=stop_id, pad_token_id=stop_id)
    return tok.decode(out[0, ids.input_ids.shape[1]:], skip_special_tokens=True)


def clean(text):
    return re.sub(r"[\[\]]", "", text) if STRIP_MARKS else text  # norm() removes all punctuation, so every bracket here is a mark


def missing(name, args, installed):
    """The ask gold would make for an installed tool that misses a required or one-of parameter, else None. Reads too: gold asks for them 7 times (flights.search date). A one-of group names its first member, as gold does for ewallet.transfer (contact_id)."""
    tool = catalog.get(name)
    if tool is None or name.split(".")[0] not in installed:
        return None
    p = tool.get("parameters", {})
    empty = lambda k: args.get(k) in (None, "", [], {})  # zaic_tools counts these as missing
    need = [k for k in p.get("required", []) if empty(k)]
    if p.get("x-one-of") and all(any(empty(k) for k in grp) for grp in p["x-one-of"]):
        need.append(p["x-one-of"][0][0])
    return {"tool_name": name, "params": need} if need else None


def run(item):
    env = ToolEnv(split, item["id"], root=ROOT)  # exactly one per sample, every call on it goes into the submission
    texts = lm.gold_texts(item) if mode == "gold" else {c: clean(asr[f"{split}/{c}"]) for c in lm.clips(item)}
    msgs = lm.build_messages(item, texts, catalog)
    t0, invalid, failed, stop, raw = time.time(), 0, 0, "max_steps", []
    try:
        for _ in range(MAX_STEPS):
            if time.time() - t0 > TIME_BUDGET:
                stop = "budget"
                break
            raw.append(generate(msgs))
            action = lm.parse_action(raw[-1])
            if action is None:  # parse_action returns None for both, and it stays as saved with the weights
                stop = "done" if '"done"' in raw[-1] else "unparsable"
                break
            call = {"name": action[0], "arguments": action[1]}
            if not env.validate(*action)["ok"]:  # validate is not logged: a call we never make costs no U penalty
                ask = missing(*action, item["context"]["installed_apps"])
                if ask and env.validate(lm.ASK, ask)["ok"]:  # gold ends on its ask too
                    env.call(lm.ASK, ask)
                    stop = "repaired"
                else:
                    invalid, stop = 1, "invalid"
                break
            if env.get_tool_calls().count(call) >= 2:  # greedy loop, gold repeats a call at most once
                stop = "repeat"
                break
            if catalog.get(call["name"], {}).get("kind") == "read" and [c["name"] for c in env.get_tool_calls()].count(call["name"]) >= 2:  # gold reads one tool 3+ times in 1 of 1,545 samples; LFM3 ver3 retried empty searches 127 times on public, then sometimes wrote on a guess
                stop = "repeat"
                break
            said = json.JSONDecoder().raw_decode(raw[-1][raw[-1].index("{"):])[0]  # the model's own JSON, so a trained "why" stays in context. Equals call for other models
            result = env.call(*action)
            msgs += [lm.asst(said), lm.result_msg(result)]
            if "query" in action[1] and not catalog[call["name"]]["parameters"].get("required") and any(v == [] for v in result.values()):  # gold never reads an empty list (0 of 234 contacts.search): the query was misheard, so list everything once and let the model pick. 25 public samples hit this (contacts, event_tickets); a full list fits MAX_RESULT_CHARS
                msgs += [lm.asst({"name": call["name"], "arguments": {}}), lm.result_msg(env.call(call["name"], {}))]
            if call["name"] == lm.ASK:  # gold always ends here
                stop = "ask"
                break
    except Exception as e:  # keep the calls already made, a crash must not lose the whole line
        failed, stop = 1, "crash"
        print(item["id"], repr(e), flush=True)
    return {"id": item["id"], "tool_calls": env.get_tool_calls(), "state_hash": env.get_state_hash(), "sec": time.time() - t0, "invalid": invalid, "failed": failed, "stop": stop, "raw": raw}


with open(f"pred_{split}_{mode}_{shard}.jsonl", "w", encoding="utf-8") as f:
    for n, item in enumerate(rows):
        line = run(item)
        if split == "train":  # scoring only: the second env replays gold for its hash, nothing here reaches a submission
            gold = item["label"]["tool_calls"]
            ref = ToolEnv(split, item["id"], root=ROOT)
            for c in gold:
                ref.call(c["name"], c["arguments"])
            line["score"] = lm.score(line["tool_calls"], gold, line["state_hash"], ref.get_state_hash(), catalog)
            line["unseen"] = bool(lm.used_apps(item) & lm.HOLDOUT_APPS)
        f.write(json.dumps(line, ensure_ascii=False) + "\n")
        f.flush()
        if n % 25 == 0:
            print(split, mode, shard, n, "/", len(rows), f"{line['sec']:.1f}s", flush=True)

In [ ]:
# ASR of every clip, as the Docker does it (submission/predict.py): one clip at a time, 4 threads, then 01c's norm() and vi_itn.py
import json, re, sys, time, unicodedata, wave
from pathlib import Path

import numpy as np
import sherpa_onnx

from settings import *

sys.path.insert(0, ASR_DIR)
from vi_itn import itn
sys.path.insert(0, LIFEMATE_DIR)
import lifemate as lm

print("LLM:", LLM, "| ASR:", ASR_DIR, "| ROOT:", ROOT)
TONES = {"\u0300", "\u0301", "\u0303", "\u0309", "\u0323"}


def old_style_tone(word):
    d = unicodedata.normalize("NFD", word)
    tones = [c for c in d if c in TONES]
    if len(tones) != 1:
        return word
    base = unicodedata.normalize("NFC", "".join(c for c in d if c not in TONES))
    if base.endswith(("oa", "oe")) or (base.endswith("uy") and not base.endswith("quy")):
        return unicodedata.normalize("NFC", base[:-2] + base[-2] + tones[0] + base[-1])
    return word


def norm(s):
    s = re.sub(r"[^\w\s]|_", " ", unicodedata.normalize("NFC", s).lower())
    return " ".join(old_style_tone(w) for w in s.split())


def read_wav(path):
    with wave.open(str(path)) as w:
        assert w.getsampwidth() == 2 and w.getnchannels() == 1, f"{path}: expected 16-bit mono"
        return w.getframerate(), np.frombuffer(w.readframes(w.getnframes()), dtype="<i2").astype(np.float32) / 32768


assert norm("Hoà, khoẻ!") == "hòa khỏe"
rec = sherpa_onnx.OfflineRecognizer.from_transducer(encoder=f"{ASR_DIR}/asr_model/encoder.int8.onnx", decoder=f"{ASR_DIR}/asr_model/decoder.int8.onnx",
                                                    joiner=f"{ASR_DIR}/asr_model/joiner.int8.onnx", tokens=f"{ASR_DIR}/asr_model/tokens.txt", num_threads=4)
rows = lm.load_split(ROOT, SPLIT)
clips = [c for x in rows[: 16 if SMOKE else None] for c in lm.clips(x)]  # infer.py's smoke takes the first 16 samples
asr, t0 = {}, time.time()
for n, c in enumerate(clips):
    s = rec.create_stream()
    s.accept_waveform(*read_wav(ROOT / SPLIT / c))
    rec.decode_stream(s)
    asr[f"{SPLIT}/{c}"] = itn(norm(s.result.text))
    if n % 500 == 0:
        print(n, "/", len(clips), f"{time.time() - t0:.0f}s", c, asr[f"{SPLIT}/{c}"], flush=True)
json.dump(asr, open(ASR_JSON, "w", encoding="utf-8"), ensure_ascii=False)
print(len(asr), "clips,", sum(not t for t in asr.values()), "empty,", f"{time.time() - t0:.0f}s")

In [ ]:
from settings import *

!CUDA_VISIBLE_DEVICES=0 python infer.py {SPLIT} asr 0 & CUDA_VISIBLE_DEVICES=1 python infer.py {SPLIT} asr 1 & wait

In [ ]:
import collections
import glob
import json

pred = [json.loads(l) for f in sorted(glob.glob(f"pred_{SPLIT}_asr_*.jsonl")) for l in open(f, encoding="utf-8")]
assert len({x["id"] for x in pred}) == len(pred)
expected = sum(1 for _ in open(ROOT / SPLIT / f"{SPLIT}.jsonl", encoding="utf-8"))
assert SMOKE or len(pred) == expected, f"{len(pred)} lines, expected {expected}"
with open("submission.jsonl", "w", encoding="utf-8") as f:
    for x in pred:
        f.write(json.dumps({k: x[k] for k in ("id", "tool_calls", "state_hash")}, ensure_ascii=False) + "\n")
print(collections.Counter(x["stop"] for x in pred).most_common(), sum(x["sec"] > 30 for x in pred), "samples over 30 s")
for x in [x for x in pred if x["stop"] == "unparsable"][:5]:
    print(x["id"], repr(x["raw"][-1][:200]))
print(f"{len(pred)} lines, {sum(x['failed'] for x in pred)} failed, {sum(x['invalid'] for x in pred)} stopped on an invalid call, slowest {max(x['sec'] for x in pred):.1f}s")
!cd "{ROOT}/{SPLIT}" && PYTHONPATH=. python -m zaic_tools --split {SPLIT} --root .. --validate /kaggle/working/submission.jsonl